In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import LeaveOneGroupOut
import warnings
warnings.filterwarnings('ignore')

TARGET_TIME = 256

class CNN2D(nn.Module):
    def __init__(self, n_classes=5, in_channels=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ELU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256),
            nn.ELU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class SpectrogramDataset(Dataset):
    def __init__(self, data_dir, labels_df):
        self.samples, self.labels = [], []
        for _, row in labels_df.iterrows():
            fname = f"sub{int(row['subject']):03d}_ses{int(row['song_id']):02d}.npy"
            fpath = os.path.join(data_dir, fname)
            if not os.path.exists(fpath):
                continue
            data = np.load(fpath).astype(np.float32)

            t = data.shape[2]
            if t >= TARGET_TIME:
                data = data[:, :, :TARGET_TIME]
            else:
                pad = np.zeros((data.shape[0], data.shape[1], TARGET_TIME - t),
                               dtype=np.float32)
                data = np.concatenate([data, pad], axis=2)

            data = (data - data.mean()) / (data.std() + 1e-8)
            self.samples.append(data)
            self.labels.append(int(row['enjoyment']) - 1)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        x = torch.tensor(self.samples[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def run():
    data_root = r"C:\Users\hiro2\Documents\EEG_project\data"
    spec_dir  = os.path.join(data_root, "spectrograms")
    labels_df = pd.read_csv(os.path.join(data_root, "features_clean.csv"))
    device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[CNN2D] device: {device}")

    dataset = SpectrogramDataset(spec_dir, labels_df)
    print(f"[CNN2D] データ数: {len(dataset)}")
    print(f"[CNN2D] 入力shape: {dataset[0][0].shape}")

    groups = labels_df['subject'].values[:len(dataset)]
    logo   = LeaveOneGroupOut()
    all_accs = []

    for fold, (tr_idx, te_idx) in enumerate(
            logo.split(range(len(dataset)),
                       [dataset[i][1].item() for i in range(len(dataset))],
                       groups)):
        tr_loader = DataLoader(
            torch.utils.data.Subset(dataset, tr_idx),
            batch_size=16, shuffle=True)
        te_loader = DataLoader(
            torch.utils.data.Subset(dataset, te_idx),
            batch_size=16)

        model     = CNN2D(n_classes=5).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(50):
            model.train()
            for x, y in tr_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for x, y in te_loader:
                x, y = x.to(device), y.to(device)
                correct += (model(x).argmax(1) == y).sum().item()
        acc = correct / len(te_idx)
        all_accs.append(acc)
        print(f"[CNN2D] Fold {fold+1:2d}: acc={acc:.3f}")

    print(f"\n[CNN2D] 平均acc: {np.mean(all_accs):.3f} ± {np.std(all_accs):.3f}")
    print(f"[CNN2D] チャンス精度: {1/5:.3f}")


if __name__ == '__main__':
    run()